# 04 - Analysis and Report Plots

Results used in the final report were generated on RunPod using `run_all.py`, `run_all_v2.py`, and `run_all_v3.py`. This notebook reloads the saved JSON artifacts and regenerates the final report figures and summary statistics from those files.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from src.utils import load_results, FIGURES_DIR, RESULTS_DIR

plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'figure.dpi': 150})

## 1. Load Saved Results

In [ ]:
results = {}
for f in RESULTS_DIR.glob('*.json'):
    results[f.stem] = load_results(f.name)
    print(f'Loaded: {f.stem}')

def has_curve(result):
    return all(k in result for k in ('cumulative_budget', 'mean_accuracy', 'se_accuracy'))

print(f'\nAvailable: {sorted(results.keys())}')

## 2. Main Report Figure: TypiClust vs Random

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for key, c, mk, label, ls in [
    ('typiclust_euclidean_b10', 'tab:blue', 'o', 'TypiClust', '-'),
    ('random_b10', 'tab:gray', 's', 'Random', '--'),
]:
    r = results[key]
    b = r['cumulative_budget']
    m = np.array(r['mean_accuracy']); s = np.array(r['se_accuracy'])
    ax.plot(b, m, f'{mk}{ls}', label=label, color=c, markersize=6)
    ax.fill_between(b, m-s, m+s, alpha=0.2, color=c)

ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CIFAR-10: TypiClust vs Random'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig1_main.png'), dpi=300, bbox_inches='tight'); plt.show()

## 3. Appendix Figures for Frameworks 1 and 2

In [ ]:
color_cycle = plt.cm.tab10.colors

# Framework 1: all reported baselines
fig, ax = plt.subplots(figsize=(10, 6))
strategies_fw1 = {}
for key, r in results.items():
    if key.startswith('fw1_') or key in ('typiclust_euclidean_b10', 'random_b10'):
        if key.startswith('fw1_'):
            name = key.replace('fw1_', '').replace('_b10', '')
        elif 'typiclust' in key:
            name = 'typiclust'
        else:
            name = 'random'
        strategies_fw1[name] = r

for i, (name, r) in enumerate(sorted(strategies_fw1.items())):
    b = r['cumulative_budget']
    m = np.array(r['mean_accuracy'])
    s = np.array(r['se_accuracy'])
    color = color_cycle[i % len(color_cycle)]
    ls = '--' if name == 'random' else '-'
    lw = 2.0 if name in ('typiclust', 'random') else 1.2
    ax.plot(b, m, f'o{ls}', label=name.capitalize(), color=color, markersize=5, linewidth=lw)
    ax.fill_between(b, m - s, m + s, alpha=0.1, color=color)

ax.set_xlabel('Cumulative Budget')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Framework 1: Fully Supervised - All Strategies (CIFAR-10)')
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig_fw1_all_strategies.png'), dpi=300, bbox_inches='tight')
plt.show()

# Framework 2: supported feature-based comparisons only
fig, ax = plt.subplots(figsize=(10, 6))
for name, key, color, marker, ls in [
    ('CoreSet', 'fw2_coreset_b10', 'tab:blue', 'o', '-'),
    ('Random', 'fw2_random_b10', 'tab:orange', 'o', '--'),
    ('TypiClust', 'fw2_typiclust_b10', 'tab:green', 'o', '-'),
]:
    r = results[key]
    b = r['cumulative_budget']
    m = np.array(r['mean_accuracy'])
    s = np.array(r['se_accuracy'])
    ax.plot(b, m, f'{marker}{ls}', label=name, color=color, markersize=6, linewidth=2)
    ax.fill_between(b, m - s, m + s, alpha=0.12, color=color)

ax.set_xlabel('Cumulative Budget')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Framework 2: Self-Supervised Embedding - Supported Feature-Based Strategies')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig_fw2_all_strategies.png'), dpi=300, bbox_inches='tight')
plt.show()

## 4. Modification Figure

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
styles = {'euclidean': ('tab:blue','o'), 'cosine': ('tab:orange','^'), 'lof': ('tab:green','s')}
for key, r in results.items():
    if 'typiclust_' in key:
        t = key.replace('typiclust_','').replace('_b10','')
        if t in styles:
            c, mk = styles[t]
            b = r['cumulative_budget']
            m = np.array(r['mean_accuracy']); s = np.array(r['se_accuracy'])
            ax.plot(b, m, f'{mk}-', label=f'TypiClust ({t})', color=c, markersize=6)
            ax.fill_between(b, m-s, m+s, alpha=0.15, color=c)
if 'random_b10' in results:
    r = results['random_b10']
    ax.plot(r['cumulative_budget'], r['mean_accuracy'], 'x--', label='Random', color='tab:gray')
ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Typicality Measure Comparison'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'fig2_modification.png'), dpi=300, bbox_inches='tight'); plt.show()

## 5. Framework 3 Sanity-Check Figure

In [ ]:
if 'fw3_random_b10' in results and 'fw3_typiclust_b10' in results:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    order = ['random', 'typiclust']
    x_pos = np.arange(len(order))
    means = [results[f'fw3_{name}_b10']['mean_accuracy'] for name in order]
    stds = [results[f'fw3_{name}_b10'].get('std_accuracy', 0.0) for name in order]
    colors = ['tab:gray', 'tab:blue']
    bars = ax.bar(x_pos, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Random', 'TypiClust'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Framework 3: FlexMatch with 10 Labels (CIFAR-10)')
    ax.grid(True, alpha=0.3, axis='y')
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f'{mean:.1f}%', ha='center', va='bottom', fontsize=11)
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'fig_fw3_flexmatch.png'), dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('FW3 result files not found.')

## 6. LaTeX-Ready Tables and Final Report Statistics

In [ ]:
# Table 1 used in the report
fw1_order = [
    ('TypiClust', 'typiclust_euclidean_b10'),
    ('Random', 'random_b10'),
    ('BADGE', 'fw1_badge_b10'),
    ('Entropy', 'fw1_entropy_b10'),
    ('Margin', 'fw1_margin_b10'),
    ('BALD', 'fw1_bald_b10'),
    ('Uncertainty', 'fw1_uncertainty_b10'),
    ('DBAL', 'fw1_dbal_b10'),
    ('CoreSet', 'fw1_coreset_b10'),
]

fw1_rows = []
for label, key in fw1_order:
    r = results[key]
    row = {'Strategy': label}
    for b, m, s in zip(r['cumulative_budget'], r['mean_accuracy'], r['se_accuracy']):
        row[f'B={b}'] = f'{m:.1f} +/- {s:.1f}'
    fw1_rows.append(row)
fw1_df = pd.DataFrame(fw1_rows)
print('Framework 1 table:')
print(fw1_df.to_string(index=False))

mod_order = [
    ('Euclidean (paper)', 'typiclust_euclidean_b10'),
    ('Cosine (control)', 'typiclust_cosine_b10'),
    ('LOF', 'typiclust_lof_b10'),
]

mod_rows = []
for label, key in mod_order:
    r = results[key]
    row = {'Typicality': label}
    for b, m, s in zip(r['cumulative_budget'], r['mean_accuracy'], r['se_accuracy']):
        row[f'B={b}'] = f'{m:.1f} +/- {s:.1f}'
    mod_rows.append(row)
mod_df = pd.DataFrame(mod_rows)
print('\nModification table:')
print(mod_df.to_string(index=False))

fw1_tc = np.array(results['typiclust_euclidean_b10']['all_accuracies'])[:, -1]
fw1_rd = np.array(results['random_b10']['all_accuracies'])[:, -1]
fw1_cs = np.array(results['fw1_coreset_b10']['all_accuracies'])[:, -1]
fw2_tc = np.array(results['fw2_typiclust_b10']['all_accuracies'])[:, -1]
fw2_rd = np.array(results['fw2_random_b10']['all_accuracies'])[:, -1]

print('\nWelch t-tests at B=50:')
for label, a, b in [
    ('FW1 TypiClust vs Random', fw1_tc, fw1_rd),
    ('FW1 TypiClust vs CoreSet', fw1_tc, fw1_cs),
    ('FW2 TypiClust vs Random', fw2_tc, fw2_rd),
]:
    t, p = stats.ttest_ind(a, b, equal_var=False)
    print(f'  {label}: t={t:.3f}, p={p:.6f}')